In [ ]:
# test api vols landed
import os
import json
import requests
import pandas as pd
from dotenv import load_dotenv
from datetime import datetime

# Charger .env
load_dotenv()
AIRLABS_API_KEY = os.getenv("AIRLABS_API_KEY")

DEPARTURE_AIRPORTS = ["FRA", "MAD", "BCN", "FCO", "ZRH"]
MAX_FLIGHTS = 10


# -----------------------------
# 1) FETCH SCHEDULES (BRUT)
# -----------------------------
def fetch_schedules(dep_airports):
    base_url = "https://airlabs.co/api/v9/schedules"
    flights = []

    for dep in dep_airports:
        params = {
            "dep_iata": dep,
            "api_key": AIRLABS_API_KEY
        }

        r = requests.get(base_url, params=params)
        data = r.json()

        if "response" not in data:
            print(f"⚠️ Aucun résultat pour {dep} — {data}")
            continue

        flights.extend(data["response"])

    return flights


# -----------------------------
# 2) FILTRAGE DES CODESHARE + flight_iata NULL
# -----------------------------
def filter_codeshares(flights):
    filtered = []
    seen_cs = set()

    for f in flights:
        fi = f.get("flight_iata")
        cs = f.get("cs_flight_iata")

        if fi is None:
            continue

        if cs is not None:
            seen_cs.add(cs)
            continue

        if fi in seen_cs:
            continue

        filtered.append(f)

    return filtered


# -----------------------------
# 3) GARDER SEULEMENT LES VOL "LANDED" SI POSSIBLE
# -----------------------------
def filter_landed(flights):
    landed = [f for f in flights if f.get("status") == "landed"]
    return landed if len(landed) > 0 else flights


# -----------------------------
# 4) LIMITER À MAX_FLIGHTS
# -----------------------------
def limit_flights(flights, max_flights):
    return flights[:max_flights]


# -----------------------------
# 5) FETCH DELAYS (toujours null mais même ordre)
# -----------------------------
def fetch_delays_raw(schedules):
    delays = {}
    for s in schedules:
        delays[s["flight_iata"]] = None
    return delays


# -----------------------------
# 6) CALCUL DES FEATURES (corrigé)
# -----------------------------
def parse_dt(dt_str):
    if not dt_str:
        return None
    return datetime.strptime(dt_str, "%Y-%m-%d %H:%M")


def compute_delay(a, b):
    if a is None or b is None:
        return None
    delay = (a - b).total_seconds() / 60.0
    return max(delay, 0)  # jamais négatif


def build_features_from_schedule(s):
    dep_time = parse_dt(s.get("dep_time"))
    arr_time = parse_dt(s.get("arr_time"))
    dep_actual = parse_dt(s.get("dep_actual"))
    arr_actual = parse_dt(s.get("arr_actual"))
    dep_estimated = parse_dt(s.get("dep_estimated"))
    arr_estimated = parse_dt(s.get("arr_estimated"))

    scheduled_hour = dep_time.hour if dep_time else None
    day_of_week = dep_time.weekday() if dep_time else None
    month = dep_time.month if dep_time else None
    is_weekend = 1 if day_of_week in [5, 6] else 0 if day_of_week is not None else None

    return {
        "airline_name": s.get("airline_iata"),
        "departure_iata": s.get("dep_iata"),
        "arrival_iata": s.get("arr_iata"),
        "scheduled_hour": scheduled_hour,
        "day_of_week": day_of_week,
        "month": month,
        "is_weekend": is_weekend,

        # Retards réels
        "departure_delay_actual": compute_delay(dep_actual, dep_time),
        "arrival_delay_actual": compute_delay(arr_actual, arr_time),

        # Retards estimés
        "departure_delay_estimated": compute_delay(dep_estimated, dep_time),
        "arrival_delay_estimated": compute_delay(arr_estimated, arr_time),

        "flight_duration_scheduled": s.get("duration"),
    }


# -----------------------------
# 7) PIPELINE COMPLET
# -----------------------------
raw = fetch_schedules(DEPARTURE_AIRPORTS)
filtered = filter_codeshares(raw)
landed = filter_landed(filtered)
limited = limit_flights(landed, MAX_FLIGHTS)

delays_raw = fetch_delays_raw(limited)
features = [build_features_from_schedule(s) for s in limited]


# -----------------------------
# 8) SAUVEGARDE DES 3 JSON
# -----------------------------
with open("airlabs_landed.json", "w") as f:
    json.dump(limited, f, indent=4)

with open("airlabs_landed_delays.json", "w") as f:
    json.dump(delays_raw, f, indent=4)

with open("airlabs_landed_training_data.json", "w") as f:
    json.dump(features, f, indent=4)

print("Fichiers sauvegardés :")
print("- airlabs_landed.json")
print("- airlabs_landed_delays.json")
print("- airlabs_landed_training_data.json")


# -----------------------------
# 9) AFFICHAGE DANS LE NOTEBOOK
# -----------------------------
df_raw = pd.DataFrame(limited)
df_delays = pd.DataFrame(list(delays_raw.items()), columns=["flight_iata", "delay_info"])
df_feat = pd.DataFrame(features)

print("=== RAW SCHEDULES ===")
display(df_raw)

print("=== RAW DELAYS ===")
display(df_delays)

print("=== FEATURES ===")
display(df_feat)


Fichiers sauvegardés :
- airlabs_landed.json
- airlabs_landed_delays.json
- airlabs_landed_training_data.json
=== RAW SCHEDULES ===


,airline_iata,airline_icao,flight_iata,flight_icao,flight_number,dep_iata,dep_icao,dep_terminal,dep_gate,dep_time,...,delayed,dep_delayed,arr_delayed,aircraft_icao,arr_time_ts,dep_time_ts,arr_estimated_ts,dep_estimated_ts,arr_actual_ts,dep_actual_ts
0,OS,AUA,OS212,AUA212,212,FRA,EDDF,1,A18,2026-04-22 15:50,...,18.0,17.0,18.0,None,1776870900,1776865800,1.776872e+09,1.776867e+09,1.776872e+09,1.776867e+09
1,LH,DLH,LH114,DLH114,114,FRA,EDDF,1,A50,2026-04-22 16:15,...,NaN,2.0,NaN,None,1776870600,1776867300,NaN,1.776867e+09,NaN,1.776867e+09
2,BT,BTI,BT2614,BTI2614,2614,FRA,EDDF,1,A24,2026-04-22 16:25,...,NaN,NaN,NaN,None,1776871800,1776867900,1.776872e+09,1.776868e+09,1.776872e+09,1.776868e+09
3,LH,DLH,LH996,DLH996,996,FRA,EDDF,1,A15,2026-04-22 16:30,...,NaN,NaN,NaN,None,1776872400,1776868200,1.776872e+09,1.776868e+09,1.776872e+09,1.776868e+09
4,LH,DLH,LH840,DLH840,840,FRA,EDDF,1,A68,2026-04-22 16:35,...,NaN,1.0,NaN,None,1776873300,1776868500,1.776873e+09,1.776869e+09,1.776873e+09,1.776869e+09
5,LH,DLH,LH254,DLH254,254,FRA,EDDF,1,A64,2026-04-22 16:40,...,NaN,NaN,NaN,None,1776873600,1776868800,1.776873e+09,NaN,1.776873e+09,NaN
6,LH,DLH,LH1078,DLH1078,1078,FRA,EDDF,1,B6,2026-04-22 16:40,...,NaN,7.0,NaN,None,1776873600,1776868800,NaN,1.776869e+09,NaN,1.776869e+09
7,LH,DLH,LH56,DLH56,56,FRA,EDDF,1,A4,2026-04-22 16:50,...,NaN,NaN,NaN,None,1776872700,1776869400,1.776872e+09,1.776869e+09,1.776872e+09,1.776869e+09
8,X5,OVA,X55049,OVA5049,5049,MAD,LEMD,2,E73,2026-04-22 16:10,...,9.0,13.0,9.0,None,1776871200,1776867000,1.776872e+09,1.776868e+09,1.776872e+09,1.776868e+09
9,NI,PGA,NI1015,PGA1015,1015,MAD,LEMD,2,D62,2026-04-22 16:20,...,7.0,7.0,NaN,None,1776872700,1776867600,1.776873e+09,1.776868e+09,1.776873e+09,1.776868e+09


=== RAW DELAYS ===


,flight_iata,delay_info
0,OS212,None
1,LH114,None
2,BT2614,None
3,LH996,None
4,LH840,None
5,LH254,None
6,LH1078,None
7,LH56,None
8,X55049,None
9,NI1015,None


=== FEATURES ===


,airline_name,departure_iata,arrival_iata,scheduled_hour,day_of_week,month,is_weekend,departure_delay_actual,arrival_delay_actual,departure_delay_estimated,arrival_delay_estimated,flight_duration_scheduled
0,OS,FRA,VIE,15,2,4,0,17.0,18.0,17.0,18.0,85
1,LH,FRA,MUC,16,2,4,0,2.0,NaN,2.0,NaN,55
2,BT,FRA,BRU,16,2,4,0,0.0,0.0,0.0,0.0,65
3,LH,FRA,AMS,16,2,4,0,0.0,0.0,0.0,0.0,70
4,LH,FRA,BLL,16,2,4,0,1.0,0.0,1.0,0.0,80
5,LH,FRA,MXP,16,2,4,0,NaN,0.0,NaN,0.0,80
6,LH,FRA,LYS,16,2,4,0,7.0,NaN,7.0,NaN,80
7,LH,FRA,BRE,16,2,4,0,0.0,0.0,0.0,0.0,55
8,X5,MAD,AGP,16,2,4,0,13.0,9.0,13.0,9.0,70
9,NI,MAD,LIS,16,2,4,0,7.0,0.0,7.0,0.0,85


In [2]:
# test api vols active or scheduled 
import os
import json
import requests
import pandas as pd
from dotenv import load_dotenv
from datetime import datetime

# Charger .env
load_dotenv()
AIRLABS_API_KEY = os.getenv("AIRLABS_API_KEY")

DEPARTURE_AIRPORTS = ["FRA", "MAD", "BCN", "FCO", "ZRH"]
MAX_FLIGHTS = 10


# -----------------------------
# 1) FETCH SCHEDULES (BRUT)
# -----------------------------
def fetch_schedules(dep_airports):
    base_url = "https://airlabs.co/api/v9/schedules"
    flights = []

    for dep in dep_airports:
        params = {
            "dep_iata": dep,
            "api_key": AIRLABS_API_KEY
        }

        r = requests.get(base_url, params=params)
        data = r.json()

        if "response" not in data:
            print(f"⚠️ Aucun résultat pour {dep} — {data}")
            continue

        flights.extend(data["response"])

    return flights


# -----------------------------
# 2) FILTRAGE DES CODESHARE + flight_iata NULL
# -----------------------------
def filter_codeshares(flights):
    filtered = []
    seen_cs = set()

    for f in flights:
        fi = f.get("flight_iata")
        cs = f.get("cs_flight_iata")

        if fi is None:
            continue

        if cs is not None:
            seen_cs.add(cs)
            continue

        if fi in seen_cs:
            continue

        filtered.append(f)

    return filtered


# -----------------------------
# 3) PRIORITÉ AUX VOL "ACTIVE", SINON "SCHEDULED"
# -----------------------------
def filter_active_or_scheduled(flights):
    active = [f for f in flights if f.get("status") == "active"]
    if len(active) >= 1:
        return active

    scheduled = [f for f in flights if f.get("status") == "scheduled"]
    return scheduled


# -----------------------------
# 4) LIMITER À MAX_FLIGHTS
# -----------------------------
def limit_flights(flights, max_flights):
    return flights[:max_flights]


# -----------------------------
# 5) FETCH DELAYS (toujours null mais même ordre)
# -----------------------------
def fetch_delays_raw(schedules):
    delays = {}
    for s in schedules:
        delays[s["flight_iata"]] = None
    return delays


# -----------------------------
# 6) CALCUL DES FEATURES
# -----------------------------
def parse_dt(dt_str):
    if not dt_str:
        return None
    return datetime.strptime(dt_str, "%Y-%m-%d %H:%M")


def compute_delay(a, b):
    if a is None or b is None:
        return None
    delay = (a - b).total_seconds() / 60.0
    return max(delay, 0)


def build_features_from_schedule(s):
    dep_time = parse_dt(s.get("dep_time"))
    arr_time = parse_dt(s.get("arr_time"))
    dep_actual = parse_dt(s.get("dep_actual"))
    arr_actual = parse_dt(s.get("arr_actual"))
    dep_estimated = parse_dt(s.get("dep_estimated"))
    arr_estimated = parse_dt(s.get("arr_estimated"))

    scheduled_hour = dep_time.hour if dep_time else None
    day_of_week = dep_time.weekday() if dep_time else None
    month = dep_time.month if dep_time else None
    is_weekend = 1 if day_of_week in [5, 6] else 0 if day_of_week is not None else None

    return {
        "airline_name": s.get("airline_iata"),
        "departure_iata": s.get("dep_iata"),
        "arrival_iata": s.get("arr_iata"),
        "scheduled_hour": scheduled_hour,
        "day_of_week": day_of_week,
        "month": month,
        "is_weekend": is_weekend,

        "departure_delay_actual": compute_delay(dep_actual, dep_time),
        "arrival_delay_actual": compute_delay(arr_actual, arr_time),

        "departure_delay_estimated": compute_delay(dep_estimated, dep_time),
        "arrival_delay_estimated": compute_delay(arr_estimated, arr_time),

        "flight_duration_scheduled": s.get("duration"),
    }


# -----------------------------
# 7) PIPELINE COMPLET
# -----------------------------
raw = fetch_schedules(DEPARTURE_AIRPORTS)
filtered = filter_codeshares(raw)
active_or_scheduled = filter_active_or_scheduled(filtered)
limited = limit_flights(active_or_scheduled, MAX_FLIGHTS)

delays_raw = fetch_delays_raw(limited)
features = [build_features_from_schedule(s) for s in limited]


# -----------------------------
# 8) SAUVEGARDE DES 3 JSON (VERSION ACTIVE)
# -----------------------------
with open("airlabs_active.json", "w") as f:
    json.dump(limited, f, indent=4)

with open("airlabs_active_delays.json", "w") as f:
    json.dump(delays_raw, f, indent=4)

with open("airlabs_active_training_data.json", "w") as f:
    json.dump(features, f, indent=4)

print("Fichiers sauvegardés :")
print("- airlabs_active.json")
print("- airlabs_active_delays.json")
print("- airlabs_active_training_data.json")


# -----------------------------
# 9) AFFICHAGE DANS LE NOTEBOOK
# -----------------------------
df_raw = pd.DataFrame(limited)
df_delays = pd.DataFrame(list(delays_raw.items()), columns=["flight_iata", "delay_info"])
df_feat = pd.DataFrame(features)

print("=== RAW ACTIVE/SCHEDULED ===")
display(df_raw)

print("=== RAW DELAYS ===")
display(df_delays)

print("=== FEATURES ===")
display(df_feat)


Fichiers sauvegardés :
- airlabs_active.json
- airlabs_active_delays.json
- airlabs_active_training_data.json
=== RAW ACTIVE/SCHEDULED ===


,airline_iata,airline_icao,flight_iata,flight_icao,flight_number,dep_iata,dep_icao,dep_terminal,dep_gate,dep_time,...,duration,delayed,dep_delayed,arr_delayed,aircraft_icao,arr_time_ts,dep_time_ts,arr_estimated_ts,dep_estimated_ts,dep_actual_ts
0,EK,UAE,EK46,UAE46,46,FRA,EDDF,2,E5,2026-04-22 15:15,...,435,29.0,58.0,29.0,None,1776889800,1776863700,1.776892e+09,1776867180,1776867180
1,SK,SAS,SK7182,SAS7182,7182,FRA,EDDF,NaN,NaN,2026-04-22 15:25,...,115,49.0,56.0,49.0,None,1776871200,1776864300,1.776874e+09,1776867660,1776867660
2,CA,CCA,CA958,CCA958,958,FRA,EDDF,1,B28,2026-04-22 16:20,...,565,59.0,5.0,59.0,None,1776901500,1776867600,1.776905e+09,1776867900,1776867900
3,LH,DLH,LH1072,DLH1072,1072,FRA,EDDF,1,A7,2026-04-22 16:25,...,110,NaN,4.0,NaN,None,1776874500,1776867900,1.776874e+09,1776868140,1776868140
4,DE,CFG,DE2314,CFG2314,2314,FRA,EDDF,1,B45,2026-04-22 16:35,...,690,NaN,NaN,NaN,None,1776909900,1776868500,1.776910e+09,1776868260,1776868260
5,EN,DLA,EN8858,DLA8858,8858,FRA,EDDF,1,B16,2026-04-22 16:50,...,100,NaN,NaN,NaN,None,1776875400,1776869400,1.776874e+09,1776869340,1776869340
6,EN,DLA,EN8064,DLA8064,8064,FRA,EDDF,1,B2,2026-04-22 16:40,...,85,4.0,9.0,4.0,None,1776873900,1776868800,1.776874e+09,1776869340,1776869340
7,LO,LOT,LO434,LOT434,434,MAD,LEMD,2,E75,2026-04-22 15:35,...,210,46.0,55.0,46.0,None,1776877500,1776864900,1.776880e+09,1776868200,1776868200
8,IB,IBE,IB339,IBE339,339,MAD,LEMD,4S,S49,2026-04-22 16:30,...,600,2.0,2.0,NaN,None,1776904200,1776868200,1.776904e+09,1776868320,1776868320
9,AV,AVA,AV17,AVA17,17,MAD,LEMD,4S,S1,2026-04-22 16:40,...,625,NaN,NaN,NaN,None,1776906300,1776868800,NaN,1776868500,1776868500


=== RAW DELAYS ===


,flight_iata,delay_info
0,EK46,None
1,SK7182,None
2,CA958,None
3,LH1072,None
4,DE2314,None
5,EN8858,None
6,EN8064,None
7,LO434,None
8,IB339,None
9,AV17,None


=== FEATURES ===


,airline_name,departure_iata,arrival_iata,scheduled_hour,day_of_week,month,is_weekend,departure_delay_actual,arrival_delay_actual,departure_delay_estimated,arrival_delay_estimated,flight_duration_scheduled
0,EK,FRA,DXB,15,2,4,0,58.0,None,58.0,29.0,435
1,SK,FRA,OSL,15,2,4,0,56.0,None,56.0,49.0,115
2,CA,FRA,PEK,16,2,4,0,5.0,None,5.0,59.0,565
3,LH,FRA,BOD,16,2,4,0,4.0,None,4.0,0.0,110
4,DE,FRA,MRU,16,2,4,0,0.0,None,0.0,0.0,690
5,EN,FRA,FLR,16,2,4,0,0.0,None,0.0,0.0,100
6,EN,FRA,GRZ,16,2,4,0,9.0,None,9.0,4.0,85
7,LO,MAD,WAW,15,2,4,0,55.0,None,55.0,46.0,210
8,IB,MAD,MCO,16,2,4,0,2.0,None,2.0,0.0,600
9,AV,MAD,MDE,16,2,4,0,0.0,None,0.0,NaN,625


In [3]:
# création de nouvelle base 
from pymongo import MongoClient
import os
from dotenv import load_dotenv

load_dotenv()

client = MongoClient(os.getenv("MONGODB_URI"))

db = client["flight_delay_history_db"]

# ---------------------------------------------------------
# Création de la nouvelle collection AirLabs
# ---------------------------------------------------------
if "airlabs_historical_landed_flights" not in db.list_collection_names():
    print("🆕 Création de la collection : airlabs_historical_landed_flights")
    db.create_collection("airlabs_historical_landed_flights")
else:
    print("✔️ La collection airlabs_historical_landed_flights existe déjà")

client.close()


🆕 Création de la collection : airlabs_historical_landed_flights


In [4]:
# suppression ancienne base 
from pymongo import MongoClient
import os
from dotenv import load_dotenv

load_dotenv()

client = MongoClient(os.getenv("MONGODB_URI"))

db = client["flight_delay_history_db"]

# ---------------------------------------------------------
# Suppression de la collection AviationStack
# ---------------------------------------------------------
if "aviationstack_historical_landed_flights" in db.list_collection_names():
    print("🗑️ Suppression de aviationstack_historical_landed_flights...")
    db.drop_collection("aviationstack_historical_landed_flights")
    print("✔️ Collection supprimée")
else:
    print("ℹ️ La collection aviationstack_historical_landed_flights n'existe pas")

client.close()


🗑️ Suppression de aviationstack_historical_landed_flights...
✔️ Collection supprimée


In [6]:
from pymongo import MongoClient
import os
from dotenv import load_dotenv

load_dotenv()

client = MongoClient(os.getenv("MONGODB_URI"))

db = client["flight_delay_history_db"]

# ---------------------------------------------------------
# Recréation de la collection AviationStack supprimée
# ---------------------------------------------------------
if "aviationstack_historical_landed_flights" not in db.list_collection_names():
    print("🆕 Recréation de la collection : aviationstack_historical_landed_flights")
    db.create_collection("aviationstack_historical_landed_flights")
else:
    print("✔️ La collection existe déjà")

client.close()



🆕 Recréation de la collection : aviationstack_historical_landed_flights


In [7]:
# copie des données aviation landed
from pymongo import MongoClient
import os
from dotenv import load_dotenv

load_dotenv()

client = MongoClient(os.getenv("MONGODB_URI"))

# ---------------------------------------------------------
# Collections source et destination
# ---------------------------------------------------------
src_db = client["flight_delay_db"]
src_col = src_db["aviationstack_flights"]

dst_db = client["flight_delay_history_db"]
dst_col = dst_db["aviationstack_historical_landed_flights"]

# ---------------------------------------------------------
# Filtre EXACT demandé
# ---------------------------------------------------------
query = {
    "flight_status": "landed",
    "arrival.actual": {"$exists": True, "$ne": None, "$ne": ""}
}

docs = list(src_col.find(query))

print(f"Documents trouvés : {len(docs)}")

# ---------------------------------------------------------
# Copie dans la collection destination
# ---------------------------------------------------------
inserted = 0

for doc in docs:
    # Nettoyage de l'ancien _id pour éviter les collisions
    if "_id" in doc:
        del doc["_id"]

    dst_col.insert_one(doc)
    inserted += 1

print(f"Documents copiés dans aviationstack_historical_landed_flights : {inserted}")

client.close()


Documents trouvés : 25
Documents copiés dans aviationstack_historical_landed_flights : 25


In [8]:
# renommage ancienne base 
from pymongo import MongoClient
import os
from dotenv import load_dotenv

load_dotenv()

client = MongoClient(os.getenv("MONGODB_URI"))

old_db = client["flight_delay_db"]
new_db = client["flight_delay_db_obsolete"]

# ---------------------------------------------------------
# Copier toutes les collections
# ---------------------------------------------------------
collections = old_db.list_collection_names()

print("Collections trouvées :", collections)

for col_name in collections:
    print(f"📁 Copie de la collection : {col_name}")

    src_col = old_db[col_name]
    dst_col = new_db[col_name]

    docs = list(src_col.find({}))

    if docs:
        # Supprimer les _id pour éviter les collisions
        for d in docs:
            d.pop("_id", None)

        dst_col.insert_many(docs)

print("✔️ Copie terminée dans flight_delay_db_obsolete")

client.close()


Collections trouvées : ['aviationstack_flights', 'clean_weather_data', 'afklm_flights', 'weather_data']
📁 Copie de la collection : aviationstack_flights
📁 Copie de la collection : clean_weather_data
📁 Copie de la collection : afklm_flights
📁 Copie de la collection : weather_data
✔️ Copie terminée dans flight_delay_db_obsolete


In [11]:
# suppression ancienne base 
from pymongo import MongoClient
import os
from dotenv import load_dotenv

load_dotenv()

client = MongoClient(os.getenv("MONGODB_URI"))

# ---------------------------------------------------------
# Suppression complète de la base flight_delay_db_obsolete
# ---------------------------------------------------------
if "flight_delay_db_obsolete" in client.list_database_names():
    print("🗑️ Suppression de la base : flight_delay_db_obsolete ...")
    client.drop_database("flight_delay_db_obsolete")
    print("✔️ Base supprimée")
else:
    print("ℹ️ La base flight_delay_db_obsolete n'existe pas")

client.close()


🗑️ Suppression de la base : flight_delay_db_obsolete ...


OperationFailure: user is not allowed to do action [dropDatabase] on [flight_delay_db_obsolete.], full error: {'ok': 0, 'errmsg': 'user is not allowed to do action [dropDatabase] on [flight_delay_db_obsolete.]', 'code': 8000, 'codeName': 'AtlasError'}